# 🧠 Smart Undersampling for Insulin Dosage Prediction

Bu notebook, **88 milyon satırlık** büyük bir veri setini dengeli bir eğitim seti haline getirmek için **Polars** kütüphanesini kullanır.

## Strategy (Strateji)

| Grup | Filtre Koşulu | Eylem |
|------|---------------|-------|
| **Positives** | `bolus > 0` | TÜMÜNÜ TUT |
| **Hard Negatives** | `bolus == 0` AND `CGM > 180` | TÜMÜNÜ TUT |
| **Easy Negatives** | `bolus == 0` AND `CGM <= 180` | Positives sayısına indir |

---

## 🔗 1. Google Drive'a Bağlan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive başarıyla bağlandı!")

## 📦 2. Setup - Kütüphaneleri Yükle

In [ ]:
# Polars kurulumu (Colab için)
!pip install polars -q

In [ ]:
import polars as pl
import time

print(f"✅ Polars version: {pl.__version__}")

## 📂 3. Dosya Yollarını Tanımla

In [ ]:
# ==============================================================================
# DOSYA YOLLARI - Google Drive üzerinden
# ==============================================================================

# Base path - Drive'daki proje klasörü
BASE_PATH = "/content/drive/MyDrive/DiabetesKD5/data/processed"

# Input ve Output dosyaları
INPUT_FILE = f"{BASE_PATH}/metabonet_processed.parquet"
OUTPUT_FILE = f"{BASE_PATH}/train_smart_balanced.parquet"

RANDOM_SEED = 42

print("=" * 60)
print("🚀 Smart Undersampling for Insulin Dosage Prediction")
print("=" * 60)
print(f"\n📂 Input:  {INPUT_FILE}")
print(f"📂 Output: {OUTPUT_FILE}")
print(f"🎲 Random Seed: {RANDOM_SEED}")

In [ ]:
# Dosyanın var olup olmadığını kontrol et
import os

if os.path.exists(INPUT_FILE):
    file_size_gb = os.path.getsize(INPUT_FILE) / (1024**3)
    print(f"✅ Input dosyası bulundu: {file_size_gb:.2f} GB")
else:
    print(f"❌ HATA: Input dosyası bulunamadı: {INPUT_FILE}")
    print("Lütfen dosya yolunu kontrol edin!")

## 📖 4. Lazy Loading - Bellek Verimli Yükleme

In [ ]:
print("📖 Lazy DataFrames oluşturuluyor...")

start_time = time.time()

# Lazy frame oluştur - veriyi belleğe YÜKLEMEZ
lf = pl.scan_parquet(INPUT_FILE)

# Her grup için lazy filtreler tanımla
lf_positives = lf.filter(pl.col("bolus") > 0)
lf_hard_negatives = lf.filter((pl.col("bolus") == 0) & (pl.col("CGM") > 180))
lf_easy_negatives = lf.filter((pl.col("bolus") == 0) & (pl.col("CGM") <= 180))

print("✅ Lazy DataFrames oluşturuldu (henüz veri yüklenmedi)")

## 📊 5. Veriyi Topla (Collect)

In [ ]:
# ==============================================================================
# POSITIVES - Tüm pozitif örnekleri topla (bolus > 0)
# ==============================================================================

print("-" * 60)
print("📊 Positives (bolus > 0) toplanıyor...")
print("-" * 60)

df_positives = lf_positives.collect()
n_positives = len(df_positives)

print(f"✅ Positives toplandı: {n_positives:,} satır")

In [ ]:
# ==============================================================================
# HARD NEGATIVES - Zor negatif örnekleri topla (bolus=0 VE CGM>180)
# ==============================================================================

print("-" * 60)
print("📊 Hard Negatives (bolus=0 AND CGM>180) toplanıyor...")
print("-" * 60)

df_hard_negatives = lf_hard_negatives.collect()
n_hard_negatives = len(df_hard_negatives)

print(f"✅ Hard Negatives toplandı: {n_hard_negatives:,} satır")

In [ ]:
# ==============================================================================
# EASY NEGATIVES - Kolay negatifleri örnekle (bolus=0 VE CGM<=180)
# ==============================================================================

print("-" * 60)
print("📊 Easy Negatives (bolus=0 AND CGM≤180) örnekleniyor...")
print("-" * 60)

# Önce toplam easy negative sayısını al (raporlama için)
n_easy_negatives_total = lf_easy_negatives.select(pl.len()).collect().item()
print(f"📈 Toplam Easy Negatives: {n_easy_negatives_total:,} satır")

# Easy negatives'i örnekle - collect() ve sample() birlikte
df_easy_negatives_sampled = lf_easy_negatives.collect().sample(
    n=n_positives,  # Pozitif sayısına eşitle
    seed=RANDOM_SEED,
    with_replacement=False
)
n_easy_negatives_sampled = len(df_easy_negatives_sampled)

print(f"✅ Easy Negatives örneklendi: {n_easy_negatives_sampled:,} satır (positives ile eşleştirildi)")

## 🔀 6. Birleştir ve Karıştır

In [ ]:
print("-" * 60)
print("🔀 DataFrame'ler birleştiriliyor ve karıştırılıyor...")
print("-" * 60)

# Üç DataFrame'i birleştir
df_combined = pl.concat([
    df_positives,
    df_hard_negatives,
    df_easy_negatives_sampled
])

print(f"📊 Birleştirilmiş veri seti boyutu: {len(df_combined):,} satır")

# Veri setini karıştır
df_final = df_combined.sample(
    fraction=1.0,  # Tüm satırları tut
    shuffle=True,
    seed=RANDOM_SEED
)

print("✅ Veri seti başarıyla karıştırıldı")

## 💾 7. Parquet Olarak Kaydet

In [ ]:
print("-" * 60)
print("💾 Parquet dosyasına kaydediliyor...")
print("-" * 60)

df_final.write_parquet(OUTPUT_FILE)

elapsed_time = time.time() - start_time

print(f"✅ Kaydedildi: {OUTPUT_FILE}")
print(f"⏱️  Toplam işlem süresi: {elapsed_time:.2f} saniye")

## 📋 8. Final Rapor

In [ ]:
# ==============================================================================
# FINAL RAPOR
# ==============================================================================

# İstatistikleri hesapla
total_rows = len(df_final)
n_positive_final = df_final.filter(pl.col("bolus") > 0).height
class_balance = (n_positive_final / total_rows) * 100

print("\n" + "=" * 60)
print("📋 FINAL RAPOR - Smart Undersampling Tamamlandı")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────────┐
│  GRUP DAĞILIMI                                          │
├─────────────────────────────────────────────────────────┤
│  ✓ Positives (bolus > 0):        {n_positives:>12,} satır  │
│  ✓ Hard Negatives (CGM > 180):   {n_hard_negatives:>12,} satır  │
│  ✓ Easy Negatives (örneklenmiş): {n_easy_negatives_sampled:>12,} satır  │
├─────────────────────────────────────────────────────────┤
│  📊 TOPLAM VERİ SETİ BOYUTU:     {total_rows:>12,} satır  │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│  SINIF DENGESİ                                          │
├─────────────────────────────────────────────────────────┤
│  Pozitif Sınıf (bolus > 0):     {class_balance:>12.2f}%        │
│  Negatif Sınıf (bolus = 0):     {100 - class_balance:>12.2f}%        │
└─────────────────────────────────────────────────────────┘

📈 Orijinal dengesizlik: ~%3 pozitif
📈 Yeni dengeli oran: %{class_balance:.2f} pozitif

🎯 LSTM eğitimi için hazır!
""")

## 🧹 9. Bellek Temizliği (Opsiyonel)

In [ ]:
# Eğitim başlamadan önce belleği temizle
del df_positives, df_hard_negatives, df_easy_negatives_sampled, df_combined

import gc
gc.collect()

print("✅ Bellek temizlendi, LSTM eğitimine hazır!")

---

## ✅ Sonraki Adım

Artık `train_smart_balanced.parquet` dosyasını LSTM modelinizi eğitmek için kullanabilirsiniz:

```python
df_train = pl.read_parquet("/content/drive/MyDrive/DiabetesKD5/data/processed/train_smart_balanced.parquet")
```